In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
# Block 2: notebook maintenance notes and cleanup backlog

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section


#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block 3: load core libraries and instantiate helpers and plotters
# Load libraries
%load_ext autoreload
%autoreload 1
%aimport Quantapp.analytics
%aimport Quantapp.visualization
%aimport Quantapp.data
%aimport Quantapp.workflows

import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.interpolate import PchipInterpolator
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter, add_sigma_reference_lines, add_mean_reference_line, add_std_annotations, add_zone_annotation, add_horizontal_zone_trace, build_time_range_buttons, build_detail_visibility_mask, build_visibility_mask
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models, MomentumAnalytics, RiskDistributionAnalytics, RiskRelativeAnalytics, SeriesTransforms, calculate_zscore, calculate_max_drawdown, gini_coefficient, calculate_window_metrics
from Quantapp.data import MacroDataClient, normalize_benchmark_tickers, load_benchmark_data, align_series_to_common_index
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)


In [ ]:
# Block 4: configure analysis parameters
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_ticker = '^IRX'  # Historical proxy used to derive the daily risk-free series
#should be a string or None
ticker_str ='NVDA'#ticker to analyze
benchmark_tickers = ['SPY']  # Benchmarks to compare against (e.g., S&P 500, Bitcoin)
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'position'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames
var_position_value = None  # Optional notional used to translate VaR / CVaR into dollar terms


In [ ]:
# Block 5: organize structural, position, swing timeframe lists

# Define strategy timeframe profiles
TIMEFRAME_PROFILES = {
    'swing': {'short': 3, 'mid': 9, 'long': 21},
    'position': {'short': 21, 'mid': 50, 'long': 200},
    'structural': {'short': 200, 'mid': 500, 'long': 1000},
}

strategy = str(trading_strategy).strip().lower()
if strategy not in TIMEFRAME_PROFILES:
    raise ValueError(
        f"Invalid trading_strategy '{trading_strategy}'. "
        f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
    )

time_frame_map = TIMEFRAME_PROFILES[strategy]
time_frame_short = time_frame_map['short']
time_frame_mid = time_frame_map['mid']
time_frame_long = time_frame_map['long']

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 6: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 7: display regime-change candlestick with Bollinger bands

#REGIME CHANGES: Candlestick with RSI and Bollinger Bands
candlestick_period = period
candlestick_view_options = [
    ('Full', None),
    ('10 Years', pd.DateOffset(years=10)),
    ('5 Years', pd.DateOffset(years=5)),
    ('3 Years', pd.DateOffset(years=3)),
    ('2 Years', pd.DateOffset(years=2)),
    ('1 Year', pd.DateOffset(years=1)),
    ('6 Months', pd.DateOffset(months=6)),
    ('3 Months', pd.DateOffset(months=3)),
]

candlestick_view_data = ticker.last(candlestick_period)
candlestick_view_end = candlestick_view_data.index.max()
candlestick_view_start = candlestick_view_data.index.min()

def build_candlestick_range(offset=None):
    start = candlestick_view_start if offset is None else max(candlestick_view_start, candlestick_view_end - offset)
    span_days = max((candlestick_view_end - start).days, 1)
    padding_days = max(10, int(span_days * 0.08))
    return [start, candlestick_view_end + pd.Timedelta(days=padding_days)]

fig = candleStickPlotter.create_candlestick_fig(
    ticker,
    period=candlestick_period,
    bollinger_window=50,
    max_drawdown_price_windows=(21, 50, 200),
    title="Candlestick with 50-Period Bollinger Bands",
)
existing_menus = list(fig.layout.updatemenus) if fig.layout.updatemenus else []
timeframe_menu = dict(
    type='dropdown',
    buttons=[
        dict(
            label=label,
            method='relayout',
            args=[{'xaxis.range': build_candlestick_range(offset)}],
        )
        for label, offset in candlestick_view_options
    ],
    direction='down',
    showactive=True,
    active=5,
    x=0.0,
    xanchor='left',
    y=1.12,
    yanchor='top',
)
fig.update_layout(
    height=1000,
    updatemenus=existing_menus + [timeframe_menu],
)
fig.update_xaxes(range=build_candlestick_range(pd.DateOffset(years=1)))
fig.add_annotation(
    text='View timeframe',
    x=0.0,
    xref='paper',
    y=1.145,
    yref='paper',
    showarrow=False,
    xanchor='left',
)
fig.add_annotation(
    text='Overlay mode',
    x=0.18,
    xref='paper',
    y=1.145,
    yref='paper',
    showarrow=False,
    xanchor='left',
)
fig

In [ ]:
# Block 8: stack candlestick, drawdown comparison, and rolling recovery time

peak_damage_window_options = [21, 50, 200]

peak_damage_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=peak_damage_window_options,
    default_window=200 if 200 in peak_damage_window_options else max(peak_damage_window_options),
)

fig = lineChartPlotter.plot_candlestick_drawdown_recovery_profile(
    price_frame=ticker,
    metrics_by_window=peak_damage_context['metrics_by_window'],
    window_options=peak_damage_context['windows'],
    default_window=peak_damage_context['default_window'],
    ticker_label=ticker_str,
    candlestick_period=period,
)
show_plotly_figure(fig)


In [ ]:
# Block 9: drawdown and recovery comparison now lives in Block 8

print('Block 9 is now combined with Block 8. Run Block 8 to view the drawdown comparison and rolling recovery-time chart.')


In [ ]:
# Block 10: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

fig = qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3],
    show=False,
)
show_plotly_figure(fig)


In [ ]:
# Block 11: analyze daily returns, skew, kurtosis, and Gini z-scores

distribution_window_options = [21, 50, 200]

distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=distribution_window_options,
    default_window=200 if 200 in distribution_window_options else max(distribution_window_options),
)

fig = lineChartPlotter.plot_distribution_shape_zscores(
    metrics_by_window=distribution_context['metrics_by_window'],
    window_options=distribution_context['windows'],
    default_window=distribution_context['default_window'],
    ticker_label=ticker_str,
)
show_plotly_figure(fig)


In [ ]:
# Block 12: compute historical close-to-close downside Value at Risk (VaR) and CVaR (Expected Shortfall)

var_window_options = [21, 50, 200]
var_confidence_levels = [0.95, 0.99]
cone_window = 200
cone_interval_levels = [0.95, 0.99]

var_context = risk_distribution_analytics.build_value_at_risk_context(
    close_series=ticker['Close'],
    windows=var_window_options,
    confidence_levels=var_confidence_levels,
    default_window=200 if 200 in var_window_options else max(var_window_options),
    default_confidence=0.95,
    position_value=var_position_value,
)

var_summary = var_context['summary_table'].copy()
if var_summary.empty:
    print('No VaR metrics available for the current series.')
else:
    rename_map = {
        'Latest VaR': 'Latest Close-to-Close VaR %',
        'Latest CVaR': 'Latest Close-to-Close CVaR %',
        'Observed Breach Rate': 'Observed Close-to-Close Downside Breach Rate %',
        'Expected Breach Rate': 'Expected Close-to-Close Downside Breach Rate %',
        'Latest Rolling Breach Rate': 'Latest Rolling Close-to-Close Downside Breach Rate %',
    }
    var_summary = var_summary.rename(columns=rename_map)
    format_map = {
        'Latest Close-to-Close VaR %': '{:.2%}',
        'Latest Close-to-Close CVaR %': '{:.2%}',
        'Observed Close-to-Close Downside Breach Rate %': '{:.2%}',
        'Expected Close-to-Close Downside Breach Rate %': '{:.2%}',
        'Latest Rolling Close-to-Close Downside Breach Rate %': '{:.2%}',
    }
    if 'Latest VaR Dollar' in var_summary.columns:
        var_summary = var_summary.rename(columns={'Latest VaR Dollar': 'Latest Close-to-Close VaR Dollar'})
        format_map['Latest Close-to-Close VaR Dollar'] = '${:,.2f}'
    if 'Latest CVaR Dollar' in var_summary.columns:
        var_summary = var_summary.rename(columns={'Latest CVaR Dollar': 'Latest Close-to-Close CVaR Dollar'})
        format_map['Latest Close-to-Close CVaR Dollar'] = '${:,.2f}'

    formatted_var_summary = var_summary.copy()
    for column, pattern in format_map.items():
        if column in formatted_var_summary.columns:
            formatted_var_summary[column] = formatted_var_summary[column].map(
                lambda value, fmt=pattern: fmt.format(value) if pd.notna(value) else value
            )

    display(formatted_var_summary)

fig = lineChartPlotter.plot_value_at_risk_profile(
    metrics_by_window=var_context['metrics_by_window'],
    window_options=var_context['windows'],
    confidence_levels=var_context['confidence_levels'],
    default_window=var_context['default_window'],
    ticker_label=ticker_str,
)
show_plotly_figure(fig)

session_cone_context = risk_distribution_analytics.build_session_probability_cone_context(
    price_frame=ticker[['Open', 'Close']],
    window=cone_window,
    interval_confidence_levels=cone_interval_levels,
    var_confidence_levels=var_confidence_levels,
)

cone_summary = session_cone_context['summary_table'].copy()
cone_metric_order = [
    'Session Open',
    'Latest Session Price',
    '95% Open-to-Close Central Range',
    '99% Open-to-Close Central Range',
    '95% Open-to-Close VaR Floor',
    '95% Open-to-Close CVaR Floor',
    '99% Open-to-Close VaR Floor',
    '99% Open-to-Close CVaR Floor',
]
if not cone_summary.empty:
    cone_summary['Metric'] = cone_summary['Metric'].replace({
        '95% Central Range': '95% Open-to-Close Central Range',
        '99% Central Range': '99% Open-to-Close Central Range',
        '95% VaR Floor': '95% Open-to-Close VaR Floor',
        '95% CVaR Floor': '95% Open-to-Close CVaR Floor',
        '99% VaR Floor': '99% Open-to-Close VaR Floor',
        '99% CVaR Floor': '99% Open-to-Close CVaR Floor',
    })
    cone_summary = cone_summary.rename(columns={
        'Lower Return': 'Lower Open-to-Close Return',
        'Upper Return': 'Upper Open-to-Close Return',
        'Lower Price': 'Lower Projected Close',
        'Upper Price': 'Upper Projected Close',
    })
    cone_summary['Metric'] = pd.Categorical(
        cone_summary['Metric'],
        categories=cone_metric_order,
        ordered=True,
    )
    cone_summary = cone_summary.sort_values('Metric').reset_index(drop=True)
    for column in ('Lower Open-to-Close Return', 'Upper Open-to-Close Return'):
        if column in cone_summary.columns:
            cone_summary[column] = cone_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower Projected Close', 'Upper Projected Close'):
        if column in cone_summary.columns:
            cone_summary[column] = cone_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )
    display(cone_summary)

cone_fig = lineChartPlotter.plot_session_probability_cone(
    cone_context=session_cone_context,
    ticker_label=ticker_str,
)
show_plotly_figure(cone_fig)


In [ ]:
# Block 13: compare annualized GARCH-family volatility models to annualized rolling volatility estimators

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Block 13 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Block 13 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Block 13 could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun Block 3 if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        return returns.rolling(window).std() * np.sqrt(252)

    volatility_frame = rolling.volatility(ohlc_frame, windows=(window,), method=method)
    return volatility_frame.iloc[:, 0]

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        return np.sqrt(ewma_variance * 252)

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        return np.sqrt(yz_variance * 252)

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

def smooth_annualized_model_vol(model_vol, window):
    # Smooth model variance over the same horizon used by the trailing realized-vol estimator.
    return np.sqrt(model_vol.pow(2).rolling(window).mean())

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

vol_model_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Annualized GARCH Family vs Close-to-Close Volatility (Raw + Smoothed)",
        "Annualized Rolling and EWMA Volatility Estimators",
        "Spread vs Close-to-Close Annualized Volatility",
        "Annualized Rolling Minus EWMA Volatility Estimators",
    ),
)
term_trace_bounds = {}
term_ranges = {}

for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }
    baseline_vol = rolling_realized_vol_map["Close-to-Close"]

    visible = term == default_vol_term
    term_trace_start = len(vol_model_fig.data)

    vol_model_fig.add_trace(
        go.Scatter(
            x=baseline_vol.index,
            y=baseline_vol,
            mode="lines",
            name=f"Close-to-Close ({window}-day)",
            line=dict(color="#1f77b4", width=2),
            visible=visible,
            showlegend=visible,
            legendgroup="close-to-close-baseline",
        ),
        row=1,
        col=1,
    )

    for model_name, _, color, dash in volatility_model_specs:
        model_vol = annualized_model_vols[model_name]
        smoothed_model_vol = smooth_annualized_model_vol(model_vol, window)
        model_spread = (model_vol - baseline_vol).dropna()
        smoothed_model_spread = (smoothed_model_vol - baseline_vol).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_vol.index,
                y=model_vol,
                mode="lines",
                name=f"Annualized {model_name}",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=visible,
                legendgroup=model_name,
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_vol.index,
                y=smoothed_model_vol,
                mode="lines",
                name=f"{model_name} Smoothed ({window}-day)",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=visible,
                legendgroup=f"{model_name}-smoothed",
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_spread.index,
                y=model_spread,
                mode="lines",
                name=f"{model_name} Spread",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-spread",
            ),
            row=3,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_spread.index,
                y=smoothed_model_spread,
                mode="lines",
                name=f"{model_name} Smoothed Spread",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-smoothed-spread",
            ),
            row=3,
            col=1,
        )

    for label, _, color, dash in rolling_realized_vol_specs:
        realized_vol = rolling_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_vol.index,
                y=realized_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"rolling-realized-{label}",
            ),
            row=2,
            col=1,
        )

        if label != "Close-to-Close":
            realized_spread = (realized_vol - baseline_vol).dropna()
            vol_model_fig.add_trace(
                go.Scatter(
                    x=realized_spread.index,
                    y=realized_spread,
                    mode="lines",
                    name=f"{label} Spread",
                    line=dict(color=color, width=2, dash=dash),
                    visible=visible,
                    showlegend=False,
                    legendgroup=f"rolling-realized-spread-{label}",
                ),
                row=3,
                col=1,
            )

    for label, method, color, dash in ewma_realized_vol_specs:
        ewma_vol = ewma_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_vol.index,
                y=ewma_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-{label}",
            ),
            row=2,
            col=1,
        )

        ewma_spread = (ewma_vol - baseline_vol).dropna()
        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_spread.index,
                y=ewma_spread,
                mode="lines",
                name=f"{label} Spread",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-spread-{label}",
            ),
            row=3,
            col=1,
        )

    for (rolling_label, _, color, _), (ewma_label, _, _, _) in zip(rolling_realized_vol_specs, ewma_realized_vol_specs):
        realized_minus_ewma = (rolling_realized_vol_map[rolling_label] - ewma_realized_vol_map[ewma_label]).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_minus_ewma.index,
                y=realized_minus_ewma,
                mode="lines",
                name=f"{rolling_label} Minus {ewma_label} ({window}-day)",
                line=dict(color=color, width=3),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"realized-minus-ewma-{rolling_label}",
            ),
            row=4,
            col=1,
        )

    term_trace_bounds[term] = (term_trace_start, len(vol_model_fig.data))

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_ranges[term] = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_ranges[term] = None

vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=3,
    col=1,
)
vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=4,
    col=1,
)

vol_buttons = []
total_traces = len(vol_model_fig.data)
for term in volatility_term_order:
    visibility = [False] * total_traces
    start, end = term_trace_bounds[term]
    for trace_idx in range(start, end):
        visibility[trace_idx] = True

    layout_updates = {
        "title": {
            "text": f"{ticker_str} Volatility Models and Estimators ({time_frame_map[term]}-Day)",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.97,
            "yanchor": "top",
        },
        "yaxis": {"title": "Annualized Volatility"},
        "yaxis2": {"title": "Annualized Volatility"},
        "yaxis3": {"title": "Spread vs Close-to-Close"},
        "yaxis4": {"title": "Rolling Minus EWMA"},
    }
    if term_ranges.get(term) is not None:
        layout_updates["xaxis"] = {"range": term_ranges[term]}
        layout_updates["xaxis2"] = {"range": term_ranges[term]}
        layout_updates["xaxis3"] = {"range": term_ranges[term]}
        layout_updates["xaxis4"] = {"range": term_ranges[term]}

    vol_buttons.append(
        dict(
            label=f"{term.title()} ({time_frame_map[term]})",
            method="update",
            args=[{"visible": visibility}, layout_updates],
        )
    )

available_ranges = [date_range for date_range in term_ranges.values() if date_range is not None]
if available_ranges:
    global_start = min(date_range[0] for date_range in available_ranges)
    global_end = max(date_range[1] for date_range in available_ranges)
    default_range = term_ranges[default_vol_term] or [global_start, global_end]
    vol_model_fig.update_xaxes(range=default_range, row=1, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=2, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=3, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=4, col=1)
    time_range_menu = dict(
        type="dropdown",
        buttons=build_time_range_buttons(global_start, global_end, axis_count=4),
        x=0.28,
        xanchor="left",
        y=1.08,
        yanchor="top",
        showactive=True,
    )
else:
    time_range_menu = None

vol_model_fig.update_layout(
    template="plotly_white",
    height=1450,
    margin=dict(t=150),
    legend=dict(x=0.01, y=0.99),
    yaxis_title="Annualized Volatility",
    yaxis2_title="Annualized Volatility",
    yaxis3_title="Spread vs Close-to-Close",
    yaxis4_title="Rolling Minus EWMA",
    title=dict(
        text=f"{ticker_str} Volatility Models and Estimators ({time_frame_map[default_vol_term]}-Day)",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=vol_buttons,
            x=0.0,
            xanchor="left",
            y=1.08,
            yanchor="top",
            showactive=True,
            active=volatility_term_order.index(default_vol_term),
        )
    ] + ([time_range_menu] if time_range_menu is not None else []),
)
show_plotly_figure(vol_model_fig)

In [ ]:
# Block 14: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))

momentum_diagnostics_context = momentum_analytics.build_momentum_window_diagnostics_context(
    close_series=ticker['Close'],
    window_sizes=window_sizes,
    highlight_windows=(7, 21, 50, 200),
    surface_years=10,
    risk_free_rate=risk_free_daily_rate,
)

momentum_diagnostic_figs = lineChartPlotter.plot_momentum_window_diagnostics(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

show_plotly_figure(momentum_diagnostic_figs['optimal_window'])
show_plotly_figure(momentum_diagnostic_figs['optimal_window_histogram'])
show_plotly_figure(momentum_diagnostic_figs['sharpe_mean_median'])
show_plotly_figure(momentum_diagnostic_figs['volatility_mean_median'])
show_plotly_figure(momentum_diagnostic_figs['sharpe_surface_3d'])


In [ ]:
# Block 15: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    ticker['Close'],
    window_pairs=window_pairs,
)

fig = lineChartPlotter.plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="200 vs 400",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.update_layout(height=850)
show_plotly_figure(fig)


In [ ]:
# Block 16: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
show_plotly_figure(fig_ticker_Seasonality_Monthly)

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
show_plotly_figure(fig_ticker_Seasonality_weekly)

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
show_plotly_figure(fig_ticker_Seasonality_daily)

In [ ]:
# Block 17: compute Sharpe/Sortino ratios and spreads

import importlib
import Quantapp.analytics.risk_relative_analytics as risk_relative_analytics_module

# Refresh the analytics class in case the notebook kernel still holds an older signature.
importlib.reload(risk_relative_analytics_module)
risk_relative_analytics = risk_relative_analytics_module.RiskRelativeAnalytics()

asset_close = ticker['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_component_map = risk_context['asset_component_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']


In [ ]:
# Block 18: plot Sharpe & Sortino efficiency with dropdown controls

long_window = time_frame_map.get('long')
default_term_label = f"{long_window}-day" if long_window is not None else max(term_config_map, key=lambda label: int(str(label).split('-')[0]))

fig = lineChartPlotter.plot_sharpe_sortino_comparison(
    term_config_map=term_config_map,
    ticker_label=ticker_str,
    default_label=default_term_label,
)
show_plotly_figure(fig)


In [ ]:
# Block 19: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = ticker['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

if not rolling_correlation_map:
    print('No rolling correlation data available for benchmark comparison.')
else:
    rolling_correlation_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    benchmark_colors = {
        symbol: rolling_correlation_palette[idx % len(rolling_correlation_palette)]
        for idx, symbol in enumerate(correlation_benchmark_order)
    }
    correlation_rows = list(rolling_correlation_map.keys())
    rolling_correlation_fig = make_subplots(
        rows=len(correlation_rows),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=[f"{time_frame_map[term]}-Day Rolling Correlation" for term in correlation_rows],
    )

    for annotation in rolling_correlation_fig.layout.annotations:
        annotation.font = dict(size=11, color='rgba(220, 220, 220, 0.90)')

    date_ranges = []
    for row_idx, term in enumerate(correlation_rows, start=1):
        row_series_map = rolling_correlation_map[term]
        row_values = []

        for symbol_idx, (symbol, correlation_series) in enumerate(row_series_map.items()):
            row_values.append(correlation_series)
            date_ranges.append((correlation_series.index.min(), correlation_series.index.max()))
            rolling_correlation_fig.add_trace(
                go.Scatter(
                    x=correlation_series.index,
                    y=correlation_series,
                    mode='lines',
                    name=symbol,
                    legendgroup=symbol,
                    line=dict(color=benchmark_colors[symbol], width=2),
                    hovertemplate=f"Benchmark: {symbol}<br>Date: %{{x|%Y-%m-%d}}<br>Rolling Correlation: %{{y:.2f}}<extra></extra>",
                    showlegend=row_idx == 1,
                ),
                row=row_idx,
                col=1,
            )
            rolling_correlation_fig.add_hline(
                y=correlation_series.mean(),
                line_dash='dot',
                line_color=benchmark_colors[symbol],
                line_width=1,
                opacity=0.7,
                row=row_idx,
                col=1,
            )
            mean_annotation_x = max(0.18, 0.985 - (symbol_idx * 0.18))
            rolling_correlation_fig.add_annotation(
                x=mean_annotation_x,
                y=correlation_series.mean(),
                xref='x domain',
                yref='y',
                text=f"{symbol} Mean {correlation_series.mean():.2f}",
                showarrow=False,
                xanchor='right',
                yanchor='bottom',
                font=dict(size=10, color=benchmark_colors[symbol]),
                row=row_idx,
                col=1,
            )

        rolling_correlation_fig.add_hline(
            y=0,
            line_dash='dash',
            line_color='rgba(180, 180, 180, 0.65)',
            line_width=1,
            opacity=0.8,
            row=row_idx,
            col=1,
        )
        rolling_correlation_fig.add_annotation(
            x=0.985,
            y=0,
            xref='x domain',
            yref='y',
            text='Zero',
            showarrow=False,
            xanchor='right',
            yanchor='top',
            font=dict(size=10, color='rgba(200, 200, 200, 0.85)'),
            row=row_idx,
            col=1,
        )

        if row_values:
            row_frame = pd.concat(row_values, axis=1)
            row_min = min(row_frame.min().min(), 0)
            row_max = max(row_frame.max().max(), 0)
            row_span = row_max - row_min
            row_padding = max(row_span * 0.08, 0.03) if pd.notna(row_span) else 0.03
            rolling_correlation_fig.update_yaxes(
                title_text='Correlation',
                range=[row_min - row_padding, row_max + row_padding],
                row=row_idx,
                col=1,
            )

    if date_ranges:
        global_start = min(start for start, _ in date_ranges)
        global_end = max(end for _, end in date_ranges)
        default_start = max(global_start, global_end - pd.DateOffset(years=3))
        for axis_idx in range(1, len(correlation_rows) + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            rolling_correlation_fig.layout[axis_name].update(range=[default_start, global_end])
        time_range_menu = dict(
            buttons=build_time_range_buttons(global_start, global_end, axis_count=len(correlation_rows)),
            direction='down',
            showactive=True,
            x=0.01,
            y=1.12,
            xanchor='left',
            yanchor='top',
            active=2,
        )
    else:
        time_range_menu = None

    rolling_correlation_fig.update_layout(
        title=f'{ticker_str} Rolling Correlation vs Benchmarks',
        template='plotly_dark',
        height=max(760, 240 * len(correlation_rows) + 140),
        margin=dict(l=50, r=40, t=90, b=40),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        updatemenus=[time_range_menu] if time_range_menu is not None else [],
    )
    rolling_correlation_fig.update_xaxes(title_text='Date', row=len(correlation_rows), col=1)
    show_plotly_figure(rolling_correlation_fig)


In [ ]:
# Block 20: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 3, 6, and 17.

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        asset_component_map=asset_component_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=time_frame_map,
    )

    default_term = 'long' if 'long' in time_frame_map else max(time_frame_map, key=time_frame_map.get)

    summary_fig = lineChartPlotter.plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_term=default_term,
    )
    show_plotly_figure(summary_fig)

    detail_fig = lineChartPlotter.plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
        default_term=default_term,
    )
    show_plotly_figure(detail_fig)
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block 21: idiosyncratic risk via Fama-French Factor Analysis
analysis_period = "max"
interval = "1d"
rolling_window = 252

ff_proxy = model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=True,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

#Plotting the Fama-French Factor Analysis Results
qp.plot_rolling_regression(rolling_results_ff5_custom, ticker_str, factor_returns_ff5_custom)
idio_fig = qp.plot_idiosyncratic_risk(rolling_results_ff5_custom, ticker_str)
